# Module 08: Fine-Tuning Data Prep & Evaluation

**Companion notebook for**: `08-fine-tuning-serving.html`

## Learning Objectives
- Prepare JSONL training data for fine-tuning
- Apply data quality filters (length, deduplication, toxicity)
- Create train/eval splits with stratification
- Configure fine-tuning with LoRA parameters
- Compute evaluation metrics (accuracy, BLEU, ROUGE)
- Run A/B comparison between base and fine-tuned models

## Prerequisites
- Python 3.10+
- `ANTHROPIC_API_KEY` environment variable set (optional -- notebook uses mock fallbacks)
- Basic understanding of supervised learning and NLP metrics

---
## Setup & Dependencies

In [ ]:
%pip install -q anthropic pydantic

In [ ]:
import os
import json
import random
import hashlib
import math
from collections import Counter
from dataclasses import dataclass
from typing import Optional

API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
USE_LIVE_API = bool(API_KEY)

print(f"Live API available: {USE_LIVE_API}")
if not USE_LIVE_API:
    print("Running in mock/simulation mode -- all outputs are simulated.")

---
## Section 1: JSONL Data Preparation

Fine-tuning requires training data in JSONL format (one JSON object per line).
Each example contains a system prompt, user message, and expected assistant response.

In [ ]:
# Generate synthetic fine-tuning data for a customer support classifier

CATEGORIES = ["billing", "technical", "account", "shipping", "returns"]

TEMPLATES = {
    "billing": [
        "I was charged twice for my last order.",
        "Can you explain the charges on my invoice?",
        "Why is my bill higher than expected this month?",
        "I need a refund for an unauthorized charge.",
        "How do I update my payment method?",
        "My coupon code didn't apply the discount.",
    ],
    "technical": [
        "The app crashes when I try to upload a photo.",
        "I can't log in even though my password is correct.",
        "The website is very slow today.",
        "How do I integrate your API with my application?",
        "Getting a 500 error when submitting the form.",
        "The search feature returns no results.",
    ],
    "account": [
        "How do I change my email address?",
        "I want to delete my account permanently.",
        "Can I merge two accounts together?",
        "How do I enable two-factor authentication?",
        "I forgot my username, how do I recover it?",
        "Can I change my subscription plan?",
    ],
    "shipping": [
        "Where is my package? It was supposed to arrive yesterday.",
        "Do you ship internationally?",
        "How long does standard shipping take?",
        "My tracking number doesn't work.",
        "Can I change the delivery address after ordering?",
        "The package arrived damaged.",
    ],
    "returns": [
        "I want to return an item I bought last week.",
        "What is your return policy?",
        "How do I get a prepaid return label?",
        "Can I exchange this for a different size?",
        "How long does it take to process a return?",
        "I received the wrong item and need to return it.",
    ],
}

SYSTEM_PROMPT = "You are a customer support classifier. Given a customer message, respond with exactly one category: billing, technical, account, shipping, or returns."


def generate_training_examples(n_per_category: int = 6) -> list[dict]:
    """Generate JSONL-format training examples."""
    examples = []
    for category, messages in TEMPLATES.items():
        for msg in messages[:n_per_category]:
            example = {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": msg},
                    {"role": "assistant", "content": category},
                ]
            }
            examples.append(example)
    return examples


raw_data = generate_training_examples()
random.seed(42)
random.shuffle(raw_data)

print(f"Generated {len(raw_data)} training examples")
print(f"Categories: {CATEGORIES}")
print(f"\nExample JSONL entry:")
print(json.dumps(raw_data[0], indent=2))

# Show category distribution
labels = [ex["messages"][2]["content"] for ex in raw_data]
dist = Counter(labels)
print(f"\nCategory distribution:")
for cat, count in sorted(dist.items()):
    bar = '#' * count
    print(f"  {cat:<12} {count:>3} {bar}")

---
## Section 2: Data Quality Filtering

Raw data often contains duplicates, too-short examples, or low-quality entries.
We apply filters to ensure training data quality.

In [ ]:
@dataclass
class QualityReport:
    total_input: int
    total_output: int
    removed_short: int
    removed_long: int
    removed_duplicates: int
    removed_toxic: int


class DataQualityFilter:
    """Apply quality filters to fine-tuning data."""
    
    def __init__(self, min_length: int = 10, max_length: int = 2000):
        self.min_length = min_length
        self.max_length = max_length
        self.toxic_keywords = [
            "kill", "hate", "stupid", "idiot",  # Simplified list
        ]
    
    def _get_user_text(self, example: dict) -> str:
        """Extract user message text from an example."""
        for msg in example.get("messages", []):
            if msg["role"] == "user":
                return msg["content"]
        return ""
    
    def _content_hash(self, example: dict) -> str:
        """Create a hash of the user message for deduplication."""
        text = self._get_user_text(example).lower().strip()
        return hashlib.md5(text.encode()).hexdigest()
    
    def _is_toxic(self, text: str) -> bool:
        """Simple keyword-based toxicity check."""
        text_lower = text.lower()
        return any(kw in text_lower for kw in self.toxic_keywords)
    
    def filter(self, examples: list[dict]) -> tuple[list[dict], QualityReport]:
        """Apply all quality filters. Returns (filtered_data, report)."""
        removed_short = 0
        removed_long = 0
        removed_dup = 0
        removed_toxic = 0
        
        seen_hashes = set()
        filtered = []
        
        for ex in examples:
            user_text = self._get_user_text(ex)
            
            # Length check
            if len(user_text) < self.min_length:
                removed_short += 1
                continue
            if len(user_text) > self.max_length:
                removed_long += 1
                continue
            
            # Deduplication
            h = self._content_hash(ex)
            if h in seen_hashes:
                removed_dup += 1
                continue
            seen_hashes.add(h)
            
            # Toxicity check
            if self._is_toxic(user_text):
                removed_toxic += 1
                continue
            
            filtered.append(ex)
        
        report = QualityReport(
            total_input=len(examples),
            total_output=len(filtered),
            removed_short=removed_short,
            removed_long=removed_long,
            removed_duplicates=removed_dup,
            removed_toxic=removed_toxic,
        )
        return filtered, report


# Add some bad examples to demonstrate filtering
bad_examples = [
    {"messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": "Hi"}, {"role": "assistant", "content": "billing"}]},  # Too short
    {"messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": "I hate your stupid product"}, {"role": "assistant", "content": "returns"}]},  # Toxic
    raw_data[0],  # Duplicate of first entry
]

data_with_issues = raw_data + bad_examples

quality_filter = DataQualityFilter(min_length=10, max_length=2000)
clean_data, report = quality_filter.filter(data_with_issues)

print("Data Quality Report")
print("=" * 40)
print(f"  Input examples:    {report.total_input}")
print(f"  Output examples:   {report.total_output}")
print(f"  Removed (short):   {report.removed_short}")
print(f"  Removed (long):    {report.removed_long}")
print(f"  Removed (dupes):   {report.removed_duplicates}")
print(f"  Removed (toxic):   {report.removed_toxic}")
print(f"  Retention rate:    {report.total_output/report.total_input:.1%}")

---
## Section 3: Train / Eval Split

Split the cleaned data into training and evaluation sets, with stratification
to ensure each category is represented proportionally in both sets.

In [ ]:
def stratified_split(data: list[dict], eval_fraction: float = 0.2,
                     seed: int = 42) -> tuple[list[dict], list[dict]]:
    """
    Split data into train/eval with stratification by label.
    Ensures each category has proportional representation.
    """
    random.seed(seed)
    
    # Group by label
    by_label: dict[str, list[dict]] = {}
    for ex in data:
        label = ex["messages"][2]["content"]
        by_label.setdefault(label, []).append(ex)
    
    train_set, eval_set = [], []
    
    for label, examples in by_label.items():
        random.shuffle(examples)
        n_eval = max(1, int(len(examples) * eval_fraction))
        eval_set.extend(examples[:n_eval])
        train_set.extend(examples[n_eval:])
    
    random.shuffle(train_set)
    random.shuffle(eval_set)
    
    return train_set, eval_set


train_data, eval_data = stratified_split(clean_data, eval_fraction=0.2)

print(f"Train set: {len(train_data)} examples")
print(f"Eval set:  {len(eval_data)} examples")
print(f"Split:     {len(train_data)/(len(train_data)+len(eval_data)):.0%} / {len(eval_data)/(len(train_data)+len(eval_data)):.0%}")

# Verify stratification
print(f"\n{'Category':<12} {'Train':>6} {'Eval':>6} {'Total':>6}")
print("-" * 35)
train_labels = Counter(ex["messages"][2]["content"] for ex in train_data)
eval_labels = Counter(ex["messages"][2]["content"] for ex in eval_data)
all_labels = sorted(set(list(train_labels.keys()) + list(eval_labels.keys())))

for label in all_labels:
    t = train_labels.get(label, 0)
    e = eval_labels.get(label, 0)
    print(f"{label:<12} {t:>6} {e:>6} {t+e:>6}")

# Write JSONL files (to demonstrate the format)
print(f"\nSample train JSONL line:")
print(json.dumps(train_data[0]))

---
## Section 4: Fine-Tune Configuration (LoRA Parameters)

LoRA (Low-Rank Adaptation) fine-tuning updates a small fraction of model parameters,
making training faster and cheaper. Here we show the configuration structure.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal


class LoRAConfig(BaseModel):
    """LoRA fine-tuning configuration."""
    # Base model
    base_model: str = "meta-llama/Llama-3.1-8B-Instruct"
    
    # LoRA-specific parameters
    lora_rank: int = Field(default=16, description="Rank of LoRA decomposition (higher = more params)")
    lora_alpha: int = Field(default=32, description="LoRA scaling factor (typically 2x rank)")
    lora_dropout: float = Field(default=0.05, ge=0.0, le=0.5)
    target_modules: list[str] = Field(
        default=["q_proj", "v_proj", "k_proj", "o_proj"],
        description="Which attention layers to apply LoRA to"
    )
    
    # Training parameters
    learning_rate: float = Field(default=2e-4, description="Peak learning rate")
    num_epochs: int = Field(default=3, ge=1, le=20)
    batch_size: int = Field(default=8, ge=1)
    gradient_accumulation_steps: int = Field(default=4, ge=1)
    warmup_ratio: float = Field(default=0.1, ge=0.0, le=0.5)
    weight_decay: float = Field(default=0.01, ge=0.0)
    max_seq_length: int = Field(default=512, ge=64)
    
    # Optimization
    optimizer: Literal["adamw", "sgd", "adafactor"] = "adamw"
    lr_scheduler: Literal["cosine", "linear", "constant"] = "cosine"
    bf16: bool = True
    
    def estimated_trainable_params(self) -> dict:
        """Estimate the number of trainable parameters."""
        # Simplified estimation for a 7B model
        hidden_dim = 4096  # Typical for 7-8B models
        n_layers = 32
        n_modules = len(self.target_modules)
        
        # Each LoRA adapter: 2 * hidden_dim * rank per module per layer
        lora_params = 2 * hidden_dim * self.lora_rank * n_modules * n_layers
        total_params = 7_000_000_000  # ~7B for base model
        
        return {
            "total_params": f"{total_params:,}",
            "trainable_params": f"{lora_params:,}",
            "trainable_pct": f"{lora_params/total_params:.4%}",
            "memory_savings": f"~{(1 - lora_params/total_params)*100:.1f}%",
        }


# Create and display configuration
config = LoRAConfig()

print("Fine-Tuning Configuration (LoRA)")
print("=" * 50)
print(json.dumps(config.model_dump(), indent=2))

print(f"\nTrainable Parameter Estimate:")
est = config.estimated_trainable_params()
for k, v in est.items():
    print(f"  {k}: {v}")

# Show different rank configurations
print(f"\n{'Rank':<8} {'Alpha':<8} {'Trainable Params':<20} {'% of Total'}")
print("-" * 55)
for rank in [4, 8, 16, 32, 64]:
    c = LoRAConfig(lora_rank=rank, lora_alpha=rank * 2)
    est = c.estimated_trainable_params()
    print(f"{rank:<8} {rank*2:<8} {est['trainable_params']:<20} {est['trainable_pct']}")

---
## Section 5: Evaluation Metrics Computation

After fine-tuning, we evaluate model quality using standard metrics:
accuracy for classification, BLEU and ROUGE for generation tasks.

In [ ]:
def compute_accuracy(predictions: list[str], references: list[str]) -> dict:
    """Compute classification accuracy and per-class metrics."""
    assert len(predictions) == len(references)
    
    correct = sum(p == r for p, r in zip(predictions, references))
    total = len(predictions)
    
    # Per-class precision and recall
    classes = sorted(set(references))
    per_class = {}
    for cls in classes:
        tp = sum(1 for p, r in zip(predictions, references) if p == cls and r == cls)
        fp = sum(1 for p, r in zip(predictions, references) if p == cls and r != cls)
        fn = sum(1 for p, r in zip(predictions, references) if p != cls and r == cls)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        per_class[cls] = {"precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3)}
    
    return {
        "accuracy": round(correct / total, 4),
        "correct": correct,
        "total": total,
        "per_class": per_class,
    }


def compute_bleu(prediction: str, reference: str, max_n: int = 4) -> float:
    """Compute a simplified BLEU score between prediction and reference."""
    pred_tokens = prediction.lower().split()
    ref_tokens = reference.lower().split()
    
    if not pred_tokens or not ref_tokens:
        return 0.0
    
    scores = []
    for n in range(1, max_n + 1):
        pred_ngrams = Counter(tuple(pred_tokens[i:i+n]) for i in range(len(pred_tokens)-n+1))
        ref_ngrams = Counter(tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens)-n+1))
        
        matches = sum((pred_ngrams & ref_ngrams).values())
        total = sum(pred_ngrams.values())
        
        if total == 0:
            scores.append(0.0)
        else:
            scores.append(matches / total)
    
    # Geometric mean with brevity penalty
    if min(scores) == 0:
        return 0.0
    
    log_avg = sum(math.log(s) for s in scores) / len(scores)
    bp = min(1.0, math.exp(1 - len(ref_tokens) / max(len(pred_tokens), 1)))
    return round(bp * math.exp(log_avg), 4)


def compute_rouge_l(prediction: str, reference: str) -> float:
    """Compute ROUGE-L (longest common subsequence) F1 score."""
    pred_tokens = prediction.lower().split()
    ref_tokens = reference.lower().split()
    
    if not pred_tokens or not ref_tokens:
        return 0.0
    
    # LCS via dynamic programming
    m, n = len(pred_tokens), len(ref_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pred_tokens[i-1] == ref_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    
    lcs_len = dp[m][n]
    precision = lcs_len / m if m > 0 else 0
    recall = lcs_len / n if n > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return round(f1, 4)


# Simulate evaluation on classification task
random.seed(42)
eval_references = [ex["messages"][2]["content"] for ex in eval_data]

# Simulated predictions (90% accurate with some errors)
eval_predictions = []
for ref in eval_references:
    if random.random() < 0.90:
        eval_predictions.append(ref)  # Correct
    else:
        wrong = random.choice([c for c in CATEGORIES if c != ref])
        eval_predictions.append(wrong)  # Wrong

metrics = compute_accuracy(eval_predictions, eval_references)

print("Classification Evaluation Metrics")
print("=" * 50)
print(f"  Overall accuracy: {metrics['accuracy']:.1%} ({metrics['correct']}/{metrics['total']})")
print(f"\n{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 45)
for cls, scores in metrics["per_class"].items():
    print(f"{cls:<12} {scores['precision']:>10.3f} {scores['recall']:>10.3f} {scores['f1']:>10.3f}")

# BLEU / ROUGE demo for generation tasks
print(f"\nGeneration Metrics Demo")
print("=" * 50)
gen_pairs = [
    ("The cat sat on the mat.", "The cat is sitting on the mat."),
    ("Machine learning uses data to learn patterns.", "Machine learning is a method that learns patterns from data."),
    ("Hello world.", "The quick brown fox jumps over the lazy dog."),
]
for pred, ref in gen_pairs:
    bleu = compute_bleu(pred, ref)
    rouge = compute_rouge_l(pred, ref)
    print(f"  Pred: {pred}")
    print(f"  Ref:  {ref}")
    print(f"  BLEU: {bleu:.4f}  ROUGE-L: {rouge:.4f}")
    print()

---
## Section 6: A/B Comparison Simulation

Compare a base model vs. a fine-tuned model on the same eval set.
This simulates the evaluation workflow used to decide whether to deploy.

In [ ]:
def simulate_model_predictions(eval_set: list[dict], accuracy: float,
                                seed: int = 0) -> list[str]:
    """Simulate model predictions at a given accuracy level."""
    random.seed(seed)
    predictions = []
    for ex in eval_set:
        true_label = ex["messages"][2]["content"]
        if random.random() < accuracy:
            predictions.append(true_label)
        else:
            wrong = random.choice([c for c in CATEGORIES if c != true_label])
            predictions.append(wrong)
    return predictions


# Simulate base model (70% accuracy) vs fine-tuned (92% accuracy)
base_preds = simulate_model_predictions(eval_data, accuracy=0.70, seed=100)
ft_preds = simulate_model_predictions(eval_data, accuracy=0.92, seed=200)

references = [ex["messages"][2]["content"] for ex in eval_data]

base_metrics = compute_accuracy(base_preds, references)
ft_metrics = compute_accuracy(ft_preds, references)

print("A/B Model Comparison")
print("=" * 60)
print(f"\n{'Metric':<20} {'Base Model':>15} {'Fine-Tuned':>15} {'Delta':>10}")
print("-" * 60)

base_acc = base_metrics["accuracy"]
ft_acc = ft_metrics["accuracy"]
delta = ft_acc - base_acc
print(f"{'Accuracy':<20} {base_acc:>14.1%} {ft_acc:>14.1%} {delta:>+9.1%}")

# Per-class comparison
print(f"\nPer-Class F1 Comparison:")
print(f"{'Class':<12} {'Base F1':>10} {'FT F1':>10} {'Delta':>10}")
print("-" * 45)
for cls in sorted(base_metrics["per_class"].keys()):
    b_f1 = base_metrics["per_class"].get(cls, {}).get("f1", 0)
    f_f1 = ft_metrics["per_class"].get(cls, {}).get("f1", 0)
    d = f_f1 - b_f1
    print(f"{cls:<12} {b_f1:>10.3f} {f_f1:>10.3f} {d:>+10.3f}")

# Decision
improvement = ft_acc - base_acc
print(f"\nDecision: ", end="")
if improvement > 0.05:
    print(f"DEPLOY fine-tuned model (+{improvement:.1%} accuracy improvement)")
elif improvement > 0:
    print(f"MARGINAL improvement (+{improvement:.1%}). Consider cost/benefit.")
else:
    print(f"KEEP base model. Fine-tuning did not improve performance.")

---
## Section 7: Training Cost Estimation

Before launching a fine-tuning job, estimate the cost based on
dataset size, model size, and training duration.

In [ ]:
def estimate_training_cost(n_examples: int, avg_tokens_per_example: int,
                           n_epochs: int, model_size_b: float) -> dict:
    """
    Estimate fine-tuning cost (simplified).
    
    Pricing is approximate and varies by provider.
    """
    total_tokens = n_examples * avg_tokens_per_example * n_epochs
    
    # Approximate cost per 1M training tokens (varies by provider/model)
    cost_per_million = {
        "small (1-3B)": 0.30,
        "medium (7-13B)": 0.80,
        "large (30-70B)": 3.00,
    }
    
    if model_size_b <= 3:
        tier = "small (1-3B)"
    elif model_size_b <= 15:
        tier = "medium (7-13B)"
    else:
        tier = "large (30-70B)"
    
    cost_per_m = cost_per_million[tier]
    estimated_cost = (total_tokens / 1_000_000) * cost_per_m
    
    # Estimate training time (very rough)
    tokens_per_second = 50000 / model_size_b  # Rough scaling
    estimated_seconds = total_tokens / tokens_per_second
    estimated_minutes = estimated_seconds / 60
    
    return {
        "total_tokens": f"{total_tokens:,}",
        "model_tier": tier,
        "cost_per_million_tokens": f"${cost_per_m:.2f}",
        "estimated_cost": f"${estimated_cost:.2f}",
        "estimated_time": f"{estimated_minutes:.0f} minutes",
        "n_examples": n_examples,
        "n_epochs": n_epochs,
    }


# Estimate costs for our dataset
scenarios = [
    (len(train_data), 80, 3, 8),    # Our small dataset on 8B model
    (1000, 150, 3, 8),              # Medium dataset on 8B model
    (10000, 200, 3, 8),             # Large dataset on 8B model
    (10000, 200, 3, 70),            # Large dataset on 70B model
]

print("Training Cost Estimates")
print("=" * 70)
print(f"{'Examples':>10} {'Tok/Ex':>8} {'Epochs':>8} {'Model':>10} {'Tokens':>15} {'Cost':>10} {'Time':>12}")
print("-" * 70)
for n_ex, tok_per_ex, epochs, model_b in scenarios:
    est = estimate_training_cost(n_ex, tok_per_ex, epochs, model_b)
    print(f"{n_ex:>10,} {tok_per_ex:>8} {epochs:>8} {model_b:>8.0f}B {est['total_tokens']:>15} {est['estimated_cost']:>10} {est['estimated_time']:>12}")

---
## Summary

In this notebook we covered the complete fine-tuning data preparation and evaluation pipeline:

1. **JSONL Data Preparation** -- Generate training data in the standard messages format (system/user/assistant) required by fine-tuning APIs.

2. **Data Quality Filtering** -- Remove short, long, duplicate, and toxic examples to ensure clean training data.

3. **Stratified Train/Eval Split** -- Split data with proportional category representation in both sets.

4. **LoRA Configuration** -- Set up parameter-efficient fine-tuning with rank, alpha, target modules, and training hyperparameters. Typical LoRA trains less than 0.1% of total parameters.

5. **Evaluation Metrics** -- Compute accuracy, precision, recall, F1 for classification. BLEU and ROUGE-L for generation tasks.

6. **A/B Comparison** -- Compare base vs. fine-tuned model on the same eval set to make deployment decisions.

7. **Cost Estimation** -- Estimate training cost and time based on dataset size, token count, and model size.

**Key insight**: Fine-tuning is only worth it when you have domain-specific data and the base model's accuracy is insufficient. Always evaluate before deploying.

**Next notebook**: Module 09 covers multi-agent orchestration.